# Fase 4: Mineração da Opinião Pública (Comentários)

Esta é a etapa mais sensível e valiosa para a modelagem de tópicos. Aqui, capturamos o texto real digitado pela audiência para entender o sentimento e os argumentos em torno da lei.

**Mecanismos de Segurança:**
* **Filtro de Engajamento:** Ignoramos vídeos com `commentCount == 0` para poupar tempo e requisições.
* **Teto de Cota:** Limitamos a extração a um máximo de 10 páginas (1.000 comentários) por vídeo. Isso impede que vídeos extremamente virais consumam toda a nossa cota diária de API de uma só vez.
* **Tratamento de Exceções:** O script está preparado para lidar com o erro `403 Forbidden` (quando o criador desativa os comentários) e o `404 Not Found` (vídeos deletados).

**Output:** `comments_dataset.csv` (Textos limpos de formatação HTML, prontos para NLP).

In [16]:
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import time
import os

In [ ]:
API_KEY = 'CHAVE_API' 
youtube = build('youtube', 'v3', developerKey=API_KEY)

In [ ]:
arquivo_entrada = "../data/complete_dataset.csv"
arquivo_comentarios = "../data/comments_dataset.csv"

MAX_PAGINAS_POR_VIDEO = 10 # Cada página tem até 100 comentários, vamos pegar no máximo 1000 por vídeos

In [19]:
df_videos = pd.read_csv(arquivo_entrada)
# Apenas vídeos que têm mais de 0 comentários
df_videos = df_videos[df_videos['commentCount'] > 0]
todos_ids_para_comentar = set(df_videos['videoId'].tolist())
print(f"Total de vídeos elegíveis para comentários: {len(todos_ids_para_comentar)}")

Total de vídeos elegíveis para comentários: 6462


In [20]:
ids_ja_processados = set()
if os.path.exists(arquivo_comentarios):
    try:
        df_comentarios_existente = pd.read_csv(arquivo_comentarios)
        ids_ja_processados = set(df_comentarios_existente['videoId'].tolist())
        print(f"Encontrados {len(ids_ja_processados)} vídeos que já tiveram seus comentários extraídos.")
    except Exception as e:
        print(f"Aviso ao ler '{arquivo_comentarios}': {e}")

ids_pendentes = list(todos_ids_para_comentar - ids_ja_processados)
print(f"Faltam processar comentários de: {len(ids_pendentes)} vídeos.\n")

if len(ids_pendentes) == 0:
    print("Todos os comentários já foram extraídos! Saindo...")
    exit()

Encontrados 5286 vídeos que já tiveram seus comentários extraídos.
Faltam processar comentários de: 1176 vídeos.



In [21]:
print("Iniciando a extração de comentários...")

try:
    for index, video_id in enumerate(ids_pendentes):
        print(f"[{index+1}/{len(ids_pendentes)}] Extraindo comentários do vídeo: {video_id}")
        
        comentarios_coletados = []
        paginas_coletadas = 0
        
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=100,
                textFormat="plainText" 
            )
            
            while request is not None and paginas_coletadas < MAX_PAGINAS_POR_VIDEO:
                response = request.execute()
                
                for item in response.get('items', []):
                    comentario = item['snippet']['topLevelComment']['snippet']
                    
                    comentarios_coletados.append({
                        'videoId': video_id,
                        'commentId': item['id'],
                        'commentAuthor': comentario.get('authorDisplayName', ''),
                        'authorChannelId': comentario.get('authorChannelId', {}).get('value', ''),
                        'commentText': comentario.get('textDisplay', ''),
                        'commentLikeCount': comentario.get('likeCount', 0),
                        'commentPublishedAt': comentario.get('publishedAt', ''),
                        'commentUpdatedAt': comentario.get('updatedAt', ''),
                        'commentReplyCount': item['snippet'].get('totalReplyCount', 0)
                    })
                
                paginas_coletadas += 1
                
                request = youtube.commentThreads().list_next(request, response)
                
            if comentarios_coletados:
                df_temp = pd.DataFrame(comentarios_coletados)
                if os.path.exists(arquivo_comentarios):
                    df_temp.to_csv(arquivo_comentarios, mode='a', header=False, index=False)
                else:
                    df_temp.to_csv(arquivo_comentarios, mode='w', header=True, index=False)
                    
        except HttpError as e:
            # Trata vídeos onde o dono desativou os comentários recentemente
            if e.resp.status == 403 and "disabled comments" in str(e).lower():
                print(f"   > Comentários desativados pelo dono do canal. Pulando...")

                colunas_vazias = ['videoId', 'commentId', 'commentAuthor', 'authorChannelId', 
                                  'text', 'commentLikeCount', 'commentPublishedAt', 
                                  'commentUpdatedAt', 'commentReplyCount']
                
                if not os.path.exists(arquivo_comentarios):
                     pd.DataFrame(columns=['videoId', 'comentario_id', 'autor', 'texto', 'curtidas', 'data_publicacao', 'total_respostas']).to_csv(arquivo_comentarios, index=False)
                pd.DataFrame([{'videoId': video_id}]).to_csv(arquivo_comentarios, mode='a', header=False, index=False)

            elif e.resp.status == 403 and "quotaExceeded" in str(e).lower():
                 print("\n[AVISO] Cota de API excedida! Parando o script principal...")
                 break # Sai do loop principal
            elif e.resp.status == 404:
                print(f"   Vídeo não encontrado (deletado/privado). Pulando...")
            else:
                print(f"   Erro HTTP ao acessar {video_id}: {e}")
                
except Exception as e:
    print(f"\nErro inesperado no pipeline: {e}")

print(f"\nMineração finalizada (ou pausada). Comentários salvos em '{arquivo_comentarios}'.")

Iniciando a extração de comentários...
[1/1176] Extraindo comentários do vídeo: 7WMmXhTTNVM
[2/1176] Extraindo comentários do vídeo: D8uK1MK22Dk
[3/1176] Extraindo comentários do vídeo: 1ujwwrZj4wk
[4/1176] Extraindo comentários do vídeo: NAxDpebnn1A
[5/1176] Extraindo comentários do vídeo: FpCgCh1GD10
[6/1176] Extraindo comentários do vídeo: rTG8odtSYak
[7/1176] Extraindo comentários do vídeo: nyysE7CFF4A
[8/1176] Extraindo comentários do vídeo: xMRfZU1L4JQ
[9/1176] Extraindo comentários do vídeo: hkdAxBWol6o
[10/1176] Extraindo comentários do vídeo: YhVdbwye3io
[11/1176] Extraindo comentários do vídeo: rxPEAh14cyc
[12/1176] Extraindo comentários do vídeo: CIgr4ju14PU
[13/1176] Extraindo comentários do vídeo: -gHanDpYqxI
[14/1176] Extraindo comentários do vídeo: aLLISIOrH3c
[15/1176] Extraindo comentários do vídeo: jhHiedLILPw
[16/1176] Extraindo comentários do vídeo: HQEhiFJtNrI
[17/1176] Extraindo comentários do vídeo: bxYYqLOqvEU
[18/1176] Extraindo comentários do vídeo: KwIzc5-a3D